In [ ]:
# Load the cleaned hourly dataset.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing

df = pd.read_csv(
    "../data/hour_cleaned.csv",
    parse_dates=["datetime"]
)

df = df.set_index("datetime").sort_index()

print(f"Original rows: {len(df):,}")
print(f"Start: {df.index.min()}")
print(f"End:   {df.index.max()}")

In [ ]:
# Build the complete hourly timeline and identify missing timestamps.

full_index = pd.date_range(
    df.index.min(),
    df.index.max(),
    freq="h"
)

missing_times = full_index.difference(df.index)

print(f"Expected hourly timestamps: {len(full_index):,}")
print(f"Observed timestamps:        {len(df):,}")
print(f"Missing timestamps:         {len(missing_times):,}")

In [ ]:
# Measure the length of every consecutive missing period.

missing_series = pd.Series(missing_times)

groups = missing_series.diff().ne(
    pd.Timedelta(hours=1)
).cumsum()

gap_lengths = missing_series.groupby(groups).size()

print(f"Missing periods: {len(gap_lengths)}")
print(f"Longest gap:     {gap_lengths.max()} hours")
print("\nGap-length distribution:")
print(gap_lengths.value_counts().sort_index().to_string())

In [ ]:
# Restore the complete hourly timeline and preserve the original observations.

df = df.reindex(full_index)
df.index.name = "datetime"

df["was_missing"] = df["cnt"].isna()

print(f"Total timestamps: {len(df):,}")
print(f"Originally missing: {df['was_missing'].sum():,}")

In [ ]:
# Create a modeling target by interpolating historical gaps while preserving a missing-data flag.

df["cnt_model"] = df["cnt"].interpolate(
    method="time",
    limit_area="inside"
)

print(f"Remaining missing values: {df['cnt_model'].isna().sum()}")
print(f"Imputed observations: {df['was_missing'].sum():,}")

In [ ]:
# Visualize an example of an imputed region to verify the preprocessing.

imputed_times = df.index[df["was_missing"]]

if len(imputed_times) > 0:

    center = imputed_times[len(imputed_times) // 2]

    window = df.loc[
        center - pd.Timedelta(hours=12):
        center + pd.Timedelta(hours=12)
    ]

    plt.figure(figsize=(14, 5))

    plt.plot(
        window.index,
        window["cnt_model"],
        marker="o",
        label="Model series"
    )

    plt.scatter(
        window.index[window["was_missing"]],
        window.loc[window["was_missing"], "cnt_model"],
        label="Imputed"
    )

    plt.title("Example of Historical Gap Imputation")
    plt.xlabel("Date")
    plt.ylabel("Bike Rentals")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Verify that the modeling series is continuous and contains no missing values.

print(f"Rows: {len(df):,}")
print(f"Missing target values: {df['cnt_model'].isna().sum()}")
print(f"Index frequency: {df.index.freq}")

In [ ]:
# Split the continuous hourly series chronologically.

y = df["cnt_model"]

n = len(y)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = y.iloc[:train_end]
val = y.iloc[train_end:val_end]
test = y.iloc[val_end:]

print(f"Train:      {train.index.min()} → {train.index.max()}")
print(f"Validation: {val.index.min()} → {val.index.max()}")
print(f"Test:       {test.index.min()} → {test.index.max()}")

print(f"\nTrain size: {len(train):,}")
print(f"Validation size: {len(val):,}")
print(f"Test size: {len(test):,}")

In [ ]:
# Define consistent metrics for comparing forecasting models.

def evaluate_model(actual, predicted):
    actual = np.asarray(actual)
    predicted = np.asarray(predicted)

    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))

    return {
        "MAE": mae,
        "RMSE": rmse
    }

# Naive baseline

In [ ]:
# Forecast every future hour using the last observed training value.

naive_pred = np.repeat(
    train.iloc[-1],
    len(val)
)

naive_metrics = evaluate_model(
    val,
    naive_pred
)

print(pd.Series(naive_metrics).round(2))

The naive model provides a minimum baseline for forecasting performance. More sophisticated models should clearly outperform this benchmark.

# Seasonal Naive

In [ ]:
# Forecast each hour using the corresponding hour from the previous day.

seasonal_naive_pred = np.tile(
    train.iloc[-24:].values,
    int(np.ceil(len(val) / 24))
)[:len(val)]

seasonal_naive_metrics = evaluate_model(
    val,
    seasonal_naive_pred
)

print(pd.Series(seasonal_naive_metrics).round(2))

# Weekly Seasonal Naive

In [ ]:
# Forecast each hour using the corresponding hour from the previous week.

weekly_naive_pred = np.tile(
    train.iloc[-168:].values,
    int(np.ceil(len(val) / 168))
)[:len(val)]

weekly_naive_metrics = evaluate_model(
    val,
    weekly_naive_pred
)

print(pd.Series(weekly_naive_metrics).round(2))

In [ ]:
# Compare simple forecasting baselines on the same validation period.

baseline_results = pd.DataFrame({
    "Naive": naive_metrics,
    "Daily Seasonal Naive": seasonal_naive_metrics,
    "Weekly Seasonal Naive": weekly_naive_metrics
}).T

baseline_results.sort_values("RMSE").round(2)

The weekly seasonal-naive model performs substantially better than both the daily seasonal-naive and naive baselines.

This confirms that the strong **168-hour weekly pattern** identified during statistical analysis is highly useful for forecasting. Classical models must therefore be evaluated against this strong baseline.

# Holt-Winters

In [ ]:
# Fit Holt-Winters with trend and weekly seasonality.

hw_model = ExponentialSmoothing(
    train,
    trend="add",
    seasonal="add",
    seasonal_periods=168
)

hw_fit = hw_model.fit()

hw_pred = hw_fit.forecast(len(val))

hw_metrics = evaluate_model(
    val,
    hw_pred
)

print(pd.Series(hw_metrics).round(2))

Holt-Winters performs substantially worse than the seasonal-naive baselines.

This indicates that a simple additive trend + weekly seasonal formulation does not capture the complex daily and weekly demand structure effectively. The weekly seasonal-naive model remains the stronger benchmark.

# ARIMA

In [ ]:
# Fit ARIMA using the differencing identified during the stationarity analysis.

arima_model = ARIMA(
    train,
    order=(2, 1, 2)
)

arima_fit = arima_model.fit()

arima_pred = arima_fit.forecast(
    steps=len(val)
)

arima_metrics = evaluate_model(
    val,
    arima_pred
)

print(pd.Series(arima_metrics).round(2))

ARIMA improves slightly over the naive baseline but performs far worse than the weekly seasonal-naive model.

The result shows that non-seasonal ARIMA does not adequately capture the strong seasonal structure in bike demand.

# SARIMA

In [ ]:
# Fit SARIMA with regular differencing and 24-hour seasonal differencing.

sarima_daily = SARIMAX(
    train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 24),
    enforce_stationarity=False,
    enforce_invertibility=False
)

sarima_daily_fit = sarima_daily.fit(
    disp=False
)

sarima_daily_pred = sarima_daily_fit.forecast(
    steps=len(val)
)

sarima_daily_metrics = evaluate_model(
    val,
    sarima_daily_pred
)

print(pd.Series(sarima_daily_metrics).round(2))

SARIMA with 24-hour seasonality performs poorly and does not outperform ARIMA or the seasonal-naive baselines.

Although daily seasonality is strong, the validation results indicate that a simple SARIMA formulation with a 24-hour seasonal period is insufficient to represent the combined daily and weekly demand structure.

# SARIMA with weekly seasonality

In [48]:
# Fit SARIMA using the stronger 168-hour weekly seasonal structure.

sarima_weekly = SARIMAX(
    train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 168),
    enforce_stationarity=False,
    enforce_invertibility=False
)

sarima_weekly_fit = sarima_weekly.fit(
    disp=False,
    maxiter=10
)

sarima_weekly_pred = sarima_weekly_fit.forecast(
    steps=len(val)
)

sarima_weekly_metrics = evaluate_model(
    val,
    sarima_weekly_pred
)

print(pd.Series(sarima_weekly_metrics).round(2))

KeyboardInterrupt: 



SARIMAX was evaluated with exogenous weather/calendar features and 24-hour seasonality. Because SARIMAX training was computationally expensive on the available hardware, the optimizer was limited to **10 iterations** for a practical benchmark.

The 10-iteration run took approximately **8-10 minutes**. Therefore, this result should be treated as a **limited computational benchmark rather than a fully optimized SARIMAX model**.

The model was included mainly to provide a classical statistical forecasting comparison against the machine-learning and deep-learning approaches.


# SARIMAX

In [50]:
# Select variables that would realistically be available when forecasting demand.

exog_columns = [
    "temp",
    "hum",
    "windspeed",
    "weathersit",
    "workingday"
]

X = df[exog_columns]

X_train = X.iloc[:train_end]
X_val = X.iloc[train_end:val_end]

print("Exogenous variables:")
print(exog_columns)

Exogenous variables:
['temp', 'hum', 'windspeed', 'weathersit', 'workingday']


In [51]:
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_train = X_train.ffill().bfill()

In [49]:
# Fit SARIMAX using weather/calendar variables and 24-hour seasonality.

sarimax_model = SARIMAX(
    train,
    exog=X_train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 24),
    enforce_stationarity=False,
    enforce_invertibility=False,
)

sarimax_fit = sarimax_model.fit(
    disp=False,
    maxiter=10 
)

sarimax_pred = sarimax_fit.forecast(
    steps=len(val),
    exog=X_val
)

sarimax_metrics = evaluate_model(
    val,
    sarimax_pred
)

print(pd.Series(sarimax_metrics).round(2))

/home/unknown/AI/.venv/lib/python3.12/site-packages/statsmodels/tsa/statespace/mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


MAE     395.25
RMSE    437.32
dtype: float64
